In [ ]:
# ======================================================================
# MODULO A — Configurazione e Parametri
# A.1 — Definizione dei parametri globali del modello e della pipeline
# ======================================================================


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  MODALITA' DI ESECUZIONE                                             ║
# ╚══════════════════════════════════════════════════════════════════════╝

# True  = Google Colab (installa dipendenze automaticamente, upload/download file)
# False = Esecuzione in locale (richiede pip install -r requirements.txt)
COLAB = True

# --- Import condizionali ---
if COLAB:
    from google.colab import files

import os

# --- Percorsi di output ---
if COLAB:
    OUTPUT_DIR = "/content/output/"
else:
    OUTPUT_DIR = "./output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Solo in locale (COLAB=False):
# False = salva automaticamente in ./output/
# True  = apre una finestra di dialogo per scegliere dove salvare il file
ASK_SAVE_PATH = False

VERBOSE = False  # se True mostra log dettagliati durante l'esecuzione


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  TOGGLE ON/OFF — Funzionalita' matematiche abilitabili               ║
# ║                                                                      ║
# ║  Ogni variabile qui sotto attiva o disattiva un passaggio            ║
# ║  specifico della pipeline. Tutte le impostazioni si applicano        ║
# ║  in modo coerente sia al backtest che al forecast futuro.            ║
# ╚══════════════════════════════════════════════════════════════════════╝

# --- 1. Rimozione outlier (winsorizing) ---
# True  = taglia i valori estremi al percentile OUTLIER_LEVEL (default 5°/95°)
# False = mantiene tutti i valori originali senza alcun filtro
REMOVE_OUTLIERS = True
OUTLIER_LEVEL   = 0.05                   # soglia: 0.05 = taglia sotto il 5° e sopra il 95° percentile

# --- 2. Rimozione zeri iniziali ---
# True  = rimuove gli zeri in testa alla serie (periodo pre-lancio prodotto)
# False = mantiene gli zeri iniziali come parte dello storico
# Nota: gli zeri interni e finali sono SEMPRE mantenuti in entrambi i casi
TRIM_LEADING_ZEROS = True

# --- 3. Calibrazione stagionale (Theil-Sen) ---
# Lista dei mesi (1-12) su cui applicare l'aggiustamento stagionale.
# Es: [8, 12] = calibra agosto e dicembre (tipici picchi/crolli domanda)
# []          = disattiva completamente la calibrazione stagionale
CALIBRATION_MONTHS = [8, 12]

# --- 4. Backtest (ottimizzazione scaling factor sul MAPE Motul) ---
# RUN_BACKTEST:
#   True  = esegue il backtest rolling-origin per trovare il quantile ottimale
#           per SKU che massimizza l'accuratezza Motul (KPI di business)
#   False = salta tutto il backtest; ogni SKU usa q = 0.5 (mediana TimesFM nativa)
#           Utile per simulazioni rapide o confronto con baseline non ottimizzata.
RUN_BACKTEST = True

# SHRINKAGE_ENABLED (rilevante solo se RUN_BACKTEST = True):
#   True  = miscela il fattore ottimale per-SKU con la mediana globale,
#           dando piu' peso alla mediana per SKU con poco storico (< 36 mesi)
#   False = usa il fattore ottimale per-SKU cosi' com'e', senza correzione
SHRINKAGE_ENABLED = True

# --- 5. Arrotondamento a multiplo d'imballo ---
# Controlla come i forecast vengono arrotondati al pack size dello SKU.
# "nearest" = arrotonda al multiplo piu' vicino
# "up"      = arrotonda sempre per eccesso
# "down"    = arrotonda sempre per difetto
# Nota: la scorta di sicurezza e' SEMPRE arrotondata per eccesso ("up"),
#       indipendentemente da questa impostazione.
ROUNDING_MODE  = "nearest"
ROUND_DECIMALS = 3                       # decimali nel risultato finale

# --- 6. Scorta di sicurezza (Safety Stock) ---
# True  = calcola classificazione ABC/XYZ e scorta di sicurezza
# False = salta il calcolo; le colonne ABC, XYZ, SafetyStock non saranno nel file di output
CALCULATE_SS = True


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  PARAMETRI NUMERICI — Orizzonti, soglie, griglie                     ║
# ╚══════════════════════════════════════════════════════════════════════╝

# --- Forecast ---
HORIZON            = 24                  # mesi da prevedere nel forecast futuro (2 anni)
MIN_HISTORY_POINTS = 6                   # minimo mesi di storico richiesti per includere uno SKU

# --- Backtest ---
HORIZON_BACKTEST   = 12                  # mesi della finestra di valutazione nel backtest
N_BACKTEST_ORIGINS = 2                   # origini di backtest (1 = singolo split, 2+ = rolling-origin con shift di 6 mesi)
# Griglia di scaling factor: passo 0.05 (grossolana) + passo 0.01 automatico (fine)
QUANTILE_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
                 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

# --- Aggiustamento di business (leva manageriale di procurement) ---
# Moltiplicatore applicato al forecast finale DOPO backtest+calibrazione e
# PRIMA dell'arrotondamento al pack. E' una decisione di business, ortogonale
# all'ottimizzazione del modello: serve a riflettere scenari esogeni (crisi,
# cambi di domanda di mercato, vincoli di stock) senza alterare la logica
# di forecasting o l'allineamento al MAPE Motul.
#   1.0   = nessun aggiustamento (default neutro)
#   < 1.0 = abbassa il forecast (es. 0.85 = -15%)
#   > 1.0 = alza il forecast   (es. 1.10 = +10%)
BUSINESS_ADJUSTMENT_FACTOR = 1.0

# --- Scorta di sicurezza ---
DEFAULT_LEAD_TIME  = 30                  # lead time di default in giorni (usato se la colonna LT manca nel file)
REORDER_PERIOD     = 30                  # periodo di riordino in giorni (fisso a 1 mese)
SS_LOOKBACK_MONTHS = 12                  # mesi di storico usati per calcolare la deviazione standard


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SOGLIE DI CLASSIFICAZIONE ABC / XYZ / Livelli di servizio           ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Soglie ABC (classificazione Pareto sui volumi cumulativi)
ABC_LIMITS = {
    "A": 0.70,  # primi 70% del volume
    "B": 0.90,  # dal 70% al 90%
    "C": 1.00   # resto
}

# Soglie XYZ (coefficiente di variazione CV = DevStd / Media)
XYZ_LIMITS = {
    "X": 0.40,  # CV <= 0.4 (domanda molto stabile)
    "Y": 0.80,  # 0.4 < CV <= 0.8 (domanda variabile)
    "Z": 999.0  # CV > 0.8 (domanda erratica)
}

# Livelli di servizio target per classe ABC/XYZ
SERVICE_LEVEL_MATRIX = {
    "AX": 0.97, "AY": 0.95, "AZ": 0.93,
    "BX": 0.91, "BY": 0.90, "BZ": 0.89,
    "CX": 0.87, "CY": 0.80, "CZ": 0.00   # 0.00 = nessuna scorta di sicurezza
}


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  MAPPATURA COLONNE EXCEL                                             ║
# ║  Modificare se il file di input ha nomi di colonna diversi           ║
# ╚══════════════════════════════════════════════════════════════════════╝

ID_COL        = "SKU"                    # chiave prodotto (codice univoco)
DESC_COL      = "Description"            # descrizione prodotto
LT_COL_NAME   = "LT"                    # lead time (giorni)
PACK_SIZE_COL = "Round"                  # multiplo d'imballo
UOM_COL       = "BUn"                    # unita' di misura


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  OUTPUT                                                              ║
# ╚══════════════════════════════════════════════════════════════════════╝

from datetime import datetime
OUTPUT_FORMAT     = "xlsx"
OUTPUT_FILE_BASE  = "Forecast and SS"
OUTPUT_SUFFIX     = datetime.now().strftime(" %Y %m %d %H_%M_%S")


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  TIMER AUTOMATICO (misura il tempo di esecuzione di ogni cella)      ║
# ║                                                                      ║
# ║  Usa un contatore sequenziale interno (_cell_seq_counter), non       ║
# ║  shell.execution_count di IPython, in modo che la numerazione        ║
# ║  rispecchi l'ordine reale di esecuzione delle celle, partendo da 1.  ║
# ╚══════════════════════════════════════════════════════════════════════╝

import time as _time
from IPython import get_ipython as _get_ipython

_cell_times = {}
_cell_start = {}
_cell_seq_counter = 0   # incrementato a ogni post_run_cell, parte da 1
_suppress_timer_print = False

def _timer_start(info=None):
    _cell_start['t'] = _time.time()

def _timer_stop(info=None):
    global _suppress_timer_print, _cell_seq_counter
    if 't' not in _cell_start:
        # pre_run_cell non ha agganciato questa cella (tipicamente la cella
        # che registra il timer): non e' misurabile, salta.
        return
    elapsed = _time.time() - _cell_start.pop('t')
    _cell_seq_counter += 1
    cell_num = _cell_seq_counter
    _cell_times[cell_num] = elapsed
    if _suppress_timer_print:
        _suppress_timer_print = False
        return
    minuti, secondi = divmod(elapsed, 60)
    if minuti >= 1:
        print(f"⏱️ Cella [{cell_num}]: {int(minuti)}m {secondi:.1f}s")
    else:
        print(f"⏱️ Cella [{cell_num}]: {secondi:.1f}s")

_ip = _get_ipython()
for _cb in list(_ip.events.callbacks.get('pre_run_cell', [])):
    if _cb.__name__ == '_timer_start':
        _ip.events.unregister('pre_run_cell', _cb)
for _cb in list(_ip.events.callbacks.get('post_run_cell', [])):
    if _cb.__name__ == '_timer_stop':
        _ip.events.unregister('post_run_cell', _cb)
_ip.events.register('pre_run_cell', _timer_start)
_ip.events.register('post_run_cell', _timer_stop)


In [ ]:
# ======================================================================
# MODULO A — Configurazione e Parametri
# A.2 — Bootstrap forecast_lib (clone in Colab, sys.path in locale)
# ======================================================================
#
# Le funzioni della pipeline vivono nel package forecast_lib/ (file .py
# affiancati al notebook). In Colab il package viene clonato da GitHub
# all'avvio: ogni nuova sessione parte da zero e prende automaticamente
# l'ultima versione di main, senza cache da invalidare.
# ======================================================================

import sys
import subprocess

REPO_URL      = "https://github.com/abaracco/Forecast-TimesFM-and-SS.git"
REPO_DIR_NAME = "Forecast-TimesFM-and-SS"   # cartella creata da `git clone`

if COLAB:
    repo_path = f"/content/{REPO_DIR_NAME}"
    if not os.path.exists(repo_path):
        print(f"📥 Clone del repository {REPO_URL} ...")
        subprocess.run(
            ["git", "clone", "-q", "--depth", "1", REPO_URL, repo_path],
            check=True,
        )
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)
else:
    # In locale: il notebook vive nella root del repo, forecast_lib/ e' a fianco.
    repo_path = os.getcwd()
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

from forecast_lib import (
    preprocessing,
    metrics,
    calibration,
    rounding,
    inventory,
    export,
    model as fl_model,
    backtest as fl_backtest,
)

print(f"✔️ forecast_lib caricato da: {repo_path}")

# Import delle librerie standard usate dall'orchestrazione del notebook
import re
import math
import numpy as np
import pandas as pd


In [ ]:
# ======================================================================
# MODULO B — Caricamento e Preprocessing
# B.1 — Caricamento file Excel (Colab: upload / Locale: finestra di dialogo)
# ======================================================================

if COLAB:
    # --- Modalita' Colab: upload tramite popup del browser ---
    print("📥 Carica il file Excel dal tuo PC...")
    uploaded = files.upload()  # apre la finestra di selezione file
    INPUT_FILE = list(uploaded.keys())[0]
else:
    # --- Modalita' locale: finestra di dialogo del sistema operativo ---
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()  # nasconde la finestra principale di tkinter
    root.attributes('-topmost', True)  # forza il dialog in primo piano su Windows
    INPUT_FILE = filedialog.askopenfilename(
        title="Seleziona il file Excel di input",
        filetypes=[("Excel files", "*.xlsx *.xls")],
        parent=root
    )
    root.destroy()
    if not INPUT_FILE:
        raise ValueError("❌ Nessun file selezionato. Operazione annullata.")
    print(f"📥 File selezionato: {INPUT_FILE}")

# Legge il primo foglio disponibile
all_sheets = pd.read_excel(INPUT_FILE, sheet_name=None)
first_sheet_name = list(all_sheets.keys())[0]
df_raw = all_sheets[first_sheet_name]

print("✔️ Foglio caricato:", first_sheet_name)
df_raw.head()


In [ ]:
# ======================================================================
# MODULO B — Caricamento e Preprocessing
# B.2..B.7 — Wide -> Long, filtro storico minimo, winsorizing
# ======================================================================
#
# Tutta la pipeline di preprocessing vive in forecast_lib.preprocessing:
#   1. wide_to_long       identifica colonne YYYY_MM, NaN -> 0, melt, parsing date
#   2. filter_min_history scarta SKU con storico inferiore a MIN_HISTORY_POINTS
#   3. apply_winsorize    clipping per percentile (se REMOVE_OUTLIERS = True)
# ======================================================================

# --- B.2..B.5: wide -> long ---
df_long, date_cols, meta_cols = preprocessing.wide_to_long(df_raw, id_col=ID_COL)
print(f"Colonne temporali trovate: {len(date_cols)}")
print(f"Colonne metadati: {meta_cols}")
print(f"Shape long: {df_long.shape}")

# --- B.6: filtro storico minimo ---
df_filtered, n_total, n_kept = preprocessing.filter_min_history(
    df_long, id_col=ID_COL, min_history_points=MIN_HISTORY_POINTS
)
print(f"\nSKU totali: {n_total}")
print(f"SKU con storico >= {MIN_HISTORY_POINTS}: {n_kept}")
print(f"Shape dopo filtro: {df_filtered.shape}")

# --- B.7: winsorizing per SKU ---
print("\n➡️ Applicazione winsorizing per SKU...")
df_filtered = preprocessing.apply_winsorize(
    df_filtered, id_col=ID_COL, level=OUTLIER_LEVEL, enabled=REMOVE_OUTLIERS
)
print("✔️ Preprocessing completato.")
df_filtered.head()


In [ ]:
# ======================================================================
# MODULO C — Serie Storiche
# C.1-C.2 — Costruzione serie storiche complete e troncate per backtest
# ======================================================================
#
# build_sku_series      -> dict {SKU: lista_valori} per il forecast finale,
#                          rimuovendo gli zeri iniziali (pre-lancio prodotto)
# build_backtest_series -> stesse serie ma TRONCATE degli ultimi
#                          HORIZON_BACKTEST mesi, piu' i target reali
# ======================================================================

# C.1 — Serie storiche complete (input al modello in produzione)
sku_series = preprocessing.build_sku_series(
    df_filtered, id_col=ID_COL, trim_leading_zeros=TRIM_LEADING_ZEROS
)
print(f"Serie generate per {len(sku_series)} SKU.")

first_sku = list(sku_series.keys())[0]
print(f"\nEsempio prima serie:")
print(f"  SKU: {first_sku}")
print(f"  Serie (primi 10 mesi): {sku_series[first_sku][:10]}")
print(f"  Lunghezza: {len(sku_series[first_sku])}")

# C.2 — Serie troncate per backtest (input al modello durante la valutazione)
backtest_series, backtest_actuals, skipped_too_short = preprocessing.build_backtest_series(
    sku_series,
    horizon_backtest=HORIZON_BACKTEST,
    min_history_points=MIN_HISTORY_POINTS,
)

print("\n📊 Riepilogo preparazione backtest:")
print(f" - SKU totali con serie storica (post-trim): {len(sku_series)}")
print(f" - SKU utilizzabili per backtest:           {len(backtest_series)}")
print(f" - SKU esclusi (serie troppo corte):        {skipped_too_short}")


In [ ]:
# ======================================================================
# MODULO D — Calibrazione Stagionale
# D.1 — Calcolo fattori di calibrazione (globali + per-SKU)
# ======================================================================
#
# I fattori sono moltiplicativi e bidirezionali: possono aumentare o
# diminuire il forecast del mese target (es. agosto, dicembre).
# Fallback: se uno SKU ha < 2 osservazioni del mese, usa il fattore globale.
#
# I fattori vengono usati nel Modulo H tramite get_calibration_factor().
# La stessa logica (su storico troncato, no leakage) e' applicata nel
# backtest del Modulo G tramite calibration.calculate_seasonality_local.
# ======================================================================

if not CALIBRATION_MONTHS:
    print("ℹ️ CALIBRATION_MONTHS vuoto: nessuna calibrazione calcolata.")
    SKU_CALIBRATION_FACTORS = {}
    GLOBAL_CALIBRATION_FACTORS = {}
else:
    SKU_CALIBRATION_FACTORS, GLOBAL_CALIBRATION_FACTORS = (
        calibration.compute_calibration_factors(
            df_filtered,
            id_col=ID_COL,
            calibration_months=CALIBRATION_MONTHS,
        )
    )

    print("Fattori globali per mesi target:",
          {m: round(GLOBAL_CALIBRATION_FACTORS[m], 4) for m in CALIBRATION_MONTHS})
    print(f"✅ Fattori per-SKU calcolati su {len(SKU_CALIBRATION_FACTORS)} SKU "
          "(bidirezionali: possono aumentare o diminuire il forecast).")


In [ ]:
# ======================================================================
# MODULO F — Modello TimesFM
# F.1 — Caricamento manuale del modello (compatibile Colab/Python 3.12)
# ======================================================================
#
# In Colab installa al volo le dipendenze (torch, einops, huggingface_hub).
# In locale vengono installate tramite requirements.txt.
# Il loader gestisce: cache HuggingFace, clone del repo TimesFM, import
# manuale del modulo PyTorch, download pesi, configurazione e device.
# ======================================================================

if COLAB:
    # Dipendenze richieste dal modello (NON il pacchetto pip "timesfm")
    !pip install -q einops huggingface_hub torch

# Carica il modello (output di log dettagliato all'interno di setup_timesfm)
model = fl_model.setup_timesfm(colab=COLAB, horizon=HORIZON)


In [ ]:
# ======================================================================
# MODULO G — Backtest e ottimizzazione scaling factor
# ======================================================================
#
# Trova lo scaling factor q ottimale per ogni SKU che massimizza
# l'accuratezza Motul su dati storici trattenuti. Pipeline:
#   1. Per ogni origine di backtest: prepara dati, lancia forecast batch
#   2. Grid search coarse (step 0.05) + fine (step 0.01) cross-origin
#   3. Shrinkage opzionale verso la mediana globale (se SHRINKAGE_ENABLED)
#
# Se RUN_BACKTEST = False: viene saltato e il forecast usa q = 0.5
# (mediana TimesFM nativa) per tutti gli SKU.
# ======================================================================

if not RUN_BACKTEST:
    print("⏩ Backtest disattivato (RUN_BACKTEST = False).")
    print("   Tutti gli SKU useranno q = 0.5 (mediana TimesFM nativa) in Modulo H.")
    df_backtest_results = fl_backtest.empty_backtest_results()
else:
    df_backtest_results = fl_backtest.run_backtest(
        model=model,
        sku_series=sku_series,
        df_filtered=df_filtered,
        id_col=ID_COL,
        pack_size_col=PACK_SIZE_COL,
        uom_col=UOM_COL,
        horizon_backtest=HORIZON_BACKTEST,
        min_history_points=MIN_HISTORY_POINTS,
        n_backtest_origins=N_BACKTEST_ORIGINS,
        quantile_grid=QUANTILE_GRID,
        calibration_months=CALIBRATION_MONTHS,
        rounding_mode=ROUNDING_MODE,
        round_decimals=ROUND_DECIMALS,
        shrinkage_enabled=SHRINKAGE_ENABLED,
        verbose=VERBOSE,
    )

    if not df_backtest_results.empty:
        mean_acc_simple = df_backtest_results["BestAccuracy"].mean()

        total_w = df_backtest_results["TotalWeight"].sum()
        if total_w > 0:
            weighted_acc_global = (
                df_backtest_results["BestAccuracy"] * df_backtest_results["TotalWeight"]
            ).sum() / total_w
        else:
            weighted_acc_global = 0.0

        print(f"\n📊 RISULTATI BACKTEST:")
        print(f" - SKU Processati: {len(df_backtest_results)}")
        print(f" - Origini backtest: {N_BACKTEST_ORIGINS}")
        print(f" - Griglia: grossolana (step 0.05) + fine (step 0.01)")
        print(f" - Shrinkage: {'attivo' if SHRINKAGE_ENABLED else 'disattivato'}")
        print(f" - Media Semplice (peso uguale per SKU):  {mean_acc_simple:.2%}")
        print(f" - MEDIA PESATA GLOBALE (peso per volume): {weighted_acc_global:.2%}  <-- KPI REALE")

        display(df_backtest_results.head())
    else:
        print("⚠️ Nessun risultato generato.")


In [ ]:
# ======================================================================
# MODULO H — Forecast Futuro
# H.1 — Esecuzione forecast point TimesFM su storico completo
# ======================================================================

print("➡️ Avvio forecast (TimesFM)")
print(f"Orizzonte scelto: {HORIZON} mesi\n")

fc_results, fc_errors = fl_model.forecast_all_skus_point(
    model=model,
    series_dict=sku_series,
    horizon=HORIZON,
    verbose=VERBOSE,
)

print("\n📊 Riepilogo forecast:")
print(f" - SKU con forecast: {len(fc_results)}")
print(f" - SKU con errori:   {len(fc_errors)}")

if fc_errors and VERBOSE:
    print("\n❌ Errori rilevati su:")
    for sku, err in fc_errors.items():
        print(f"   SKU {sku}: {err}")
elif not fc_errors:
    print("✔️ Nessun errore nei forecast.")


In [ ]:
# ======================================================================
# MODULO H — Forecast Futuro
# H.2 — Pipeline completa: scaling -> calibrazione -> business adj. -> arrotondamento
# ======================================================================
#
# Per ogni SKU applichiamo, nell'ordine:
#   1. Scaling factor ottimale (dal backtest, o 0.5 se RUN_BACKTEST = False)
#   2. Calibrazione stagionale (fattori dal Modulo D)
#   3. Aggiustamento di business (BUSINESS_ADJUSTMENT_FACTOR, leva manageriale)
#   4. Arrotondamento al multiplo d'imballo (ROUNDING_MODE)
# ======================================================================

# Timeline futura: parte dal mese successivo all'ultima data storica
last_date_global = df_filtered["Date"].max()
print("Ultima data storica globale:", last_date_global)

start_date_global = last_date_global + pd.offsets.MonthBegin(1)
future_dates_global = [
    start_date_global + pd.offsets.MonthBegin(i)
    for i in range(HORIZON)
]

# Mappa SKU -> scaling factor ottimale (dal backtest)
best_q_map = {}
if not df_backtest_results.empty:
    best_q_map = dict(zip(df_backtest_results["SKU"], df_backtest_results["BestQuantile"]))
else:
    print("⚠️ Backtest vuoto: tutti gli SKU useranno q = 0.5.")

def get_best_quantile_for_sku(sku):
    """Restituisce lo scaling factor ottimale per lo SKU, o 0.5 se non disponibile."""
    q = best_q_map.get(sku, 0.5)
    try:
        return float(q)
    except Exception:
        return 0.5

# Mappa SKU -> pack size (per arrotondamento)
meta_pack = (
    df_filtered[[ID_COL, PACK_SIZE_COL, UOM_COL]]
    .drop_duplicates()
    .set_index(ID_COL)
)

def get_pack_for_sku(sku):
    """Restituisce il pack size per lo SKU, default 1.0 se mancante o non numerico."""
    if sku not in meta_pack.index:
        return 1.0
    v = meta_pack.loc[sku, PACK_SIZE_COL]
    if pd.isna(v) or v in ("", None):
        return 1.0
    try:
        return float(v)
    except Exception:
        return 1.0

# Pipeline completa SKU per SKU
fc_rows = []

print("➡️ Pipeline completa: scaling -> calibrazione -> business adj. -> arrotondamento")

for sku, base_fc in fc_results.items():
    base_fc = np.array(base_fc, dtype=float)

    # 1. Scaling factor ottimale (dal backtest, o 0.5 come default)
    q = get_best_quantile_for_sku(sku)
    scale = q / 0.5  # fattore moltiplicativo rispetto alla mediana

    # 2. Applica scaling al forecast base
    fc_scaled = base_fc * scale

    # 3. Pack size per arrotondamento
    pack = get_pack_for_sku(sku)

    # Allinea lunghezza forecast e date future
    steps = min(len(fc_scaled), len(future_dates_global))

    for i in range(steps):
        date = future_dates_global[i]
        month = int(date.month)

        # 4. Calibrazione stagionale
        factor = calibration.get_calibration_factor(
            sku, month,
            sku_factors=SKU_CALIBRATION_FACTORS,
            global_factors=GLOBAL_CALIBRATION_FACTORS,
            calibration_months=CALIBRATION_MONTHS,
        )
        fc_calibrated = fc_scaled[i] * factor

        # 5. Aggiustamento di business (leva manageriale, ortogonale al modello)
        fc_adjusted = fc_calibrated * BUSINESS_ADJUSTMENT_FACTOR

        # 6. Arrotondamento al multiplo d'imballo
        fc_final = rounding.round_to_pack(
            value=fc_adjusted,
            pack=pack,
            mode=ROUNDING_MODE,
            decimals=ROUND_DECIMALS,
        )

        fc_rows.append({
            ID_COL: sku,
            "Date": date,
            "Period": date.strftime("%Y_%m"),
            "Forecast": float(fc_final),
            "BestQuantile": q,
        })

df_fc_long = pd.DataFrame(fc_rows)

print("Anteprima forecast LONG (valori finali: scalato + calibrato + arrotondato):")
df_fc_long.head()


In [ ]:
# ======================================================================
# MODULO H — Forecast Futuro
# H.3 — Conversione forecast da formato LONG a WIDE
# ======================================================================

print("➡️ Conversione forecast LONG -> WIDE (valori finali)...")

df_fc_wide = export.build_forecast_wide(df_fc_long, id_col=ID_COL)

print("Anteprima forecast WIDE (finale):")
df_fc_wide.head()


In [ ]:
# ======================================================================
# MODULO I — Pianificazione Inventario (ABC, XYZ, Scorta di Sicurezza)
# ======================================================================
#
# Calcolo ABC (Pareto sui volumi) + XYZ (CV) + scorta di sicurezza
# (SS = Z * sigma * sqrt((LT + ReorderPeriod) / 30), arrotondata SEMPRE
# per eccesso al pack indipendentemente da ROUNDING_MODE).
# ======================================================================

if CALCULATE_SS:
    df_inventory = inventory.calculate_inventory_logic(
        df_filtered, df_raw,
        id_col=ID_COL,
        lt_col_name=LT_COL_NAME,
        pack_size_col=PACK_SIZE_COL,
        abc_limits=ABC_LIMITS,
        xyz_limits=XYZ_LIMITS,
        service_level_matrix=SERVICE_LEVEL_MATRIX,
        ss_lookback_months=SS_LOOKBACK_MONTHS,
        default_lead_time=DEFAULT_LEAD_TIME,
        reorder_period=REORDER_PERIOD,
        round_decimals=ROUND_DECIMALS,
    )
    print("✔️ Inventario calcolato.")
    print("\nEsempio classificazione e scorta di sicurezza:")
    display(df_inventory.head())
else:
    df_inventory = None
    print("ℹ️ CALCULATE_SS = False: ABC/XYZ/SafetyStock non calcolati.")


In [ ]:
# ======================================================================
# MODULO J — Output Finale
# J.1 — Costruzione tabella finale e export Excel
# ======================================================================

print("➡️ Costruzione tabella finale (Metadati + Inventario + Storico + Forecast)...")

out_final = export.build_final_table(
    df_filtered, df_fc_wide, df_inventory,
    id_col=ID_COL, desc_col=DESC_COL,
    pack_size_col=PACK_SIZE_COL, uom_col=UOM_COL,
)

print(f"✔️ Tabella finale pronta. Shape: {out_final.shape}")

# Export Excel
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE_BASE + OUTPUT_SUFFIX + ".xlsx")
export.save_excel(out_final, output_path)
print(f"✔️ File salvato in {output_path}")

if COLAB:
    # Modalita' Colab: download automatico nel browser
    files.download(output_path)
else:
    # Modalita' locale: salvataggio nella cartella di output
    if ASK_SAVE_PATH:
        # Apre finestra di dialogo per scegliere dove salvare una copia
        import shutil
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)  # forza il dialog in primo piano su Windows
        save_path = filedialog.asksaveasfilename(
            title="Salva il file Excel di output",
            defaultextension=".xlsx",
            initialfile=OUTPUT_FILE_BASE + OUTPUT_SUFFIX + ".xlsx",
            filetypes=[("Excel files", "*.xlsx")],
            parent=root
        )
        root.destroy()
        if save_path:
            shutil.copy2(output_path, save_path)
            print(f"✔️ Copia salvata in: {save_path}")
        else:
            print(f"ℹ️ Salvataggio annullato. Il file resta in: {os.path.abspath(output_path)}")
    else:
        print(f"📁 File di output: {os.path.abspath(output_path)}")

# === RIEPILOGO TEMPI DI ESECUZIONE ===
# Impedisci a _timer_stop di stampare per questa cella (il tempo e' incluso nel riepilogo)
_suppress_timer_print = True

# Stima il tempo della cella corrente (il callback _timer_stop non e' ancora scattato)
_this_cell_elapsed = _time.time() - _cell_start.get('t', _time.time())

if _cell_times or _this_cell_elapsed > 0:
    _this_cell_num = max(_cell_times.keys()) + 1 if _cell_times else 1
    totale = sum(_cell_times.values()) + _this_cell_elapsed
    minuti_tot, secondi_tot = divmod(totale, 60)
    print(f"\n{'='*50}")
    print(f"⏱️ RIEPILOGO TEMPI DI ESECUZIONE")
    print(f"{'='*50}")
    for cella, t in sorted(_cell_times.items()):
        m, s = divmod(t, 60)
        if m >= 1:
            print(f"  Cella [{cella:>2}]: {int(m)}m {s:>5.1f}s")
        else:
            print(f"  Cella [{cella:>2}]:    {s:>5.1f}s")
    _m, _s = divmod(_this_cell_elapsed, 60)
    if _m >= 1:
        print(f"  Cella [{_this_cell_num:>2}]: {int(_m)}m {_s:>5.1f}s")
    else:
        print(f"  Cella [{_this_cell_num:>2}]:    {_s:>5.1f}s")
    print(f"{chr(9472)*50}")
    if minuti_tot >= 1:
        print(f"  TOTALE:     {int(minuti_tot)}m {secondi_tot:.1f}s")
    else:
        print(f"  TOTALE:     {secondi_tot:.1f}s")
    print(f"{'='*50}")
